In [1]:
# ================= Ablation: Expected cases averted vs budget B =================
# Compares FRC-targeted vs EdgeBetweenness-targeted vaccination on synthetic graphs
# (ER, WS, BA, PLC). "Expected" = average across multiple seed sets.
# ------------------------------------------------------------------------------

import os, math, random
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from typing import Dict, Tuple, List

# ------------------------------ Repro / config --------------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

OUTDIR = "images"
os.makedirs(OUTDIR, exist_ok=True)

# Network sizes (as in manuscript)
N = 1000
P_ER = 0.003
WS_K, WS_P = 10, 0.1
BA_M = 3
PLC_M, PLC_P = 3, 0.05

# Epidemic timeline
T_MAX = 100.0
N_T = 500
TIME = np.linspace(0, T_MAX, N_T)
DT = float(TIME[1] - TIME[0])

# SIR params
GAMMA = 0.1
ALPHA = 1.0            # beta_ij = beta0 * exp(ALPHA * z_e_std)
BETA0_TRUE = 0.3       # target baseline; may be bumped up by threshold calibration

# Intervention budgets (people vaccinated before the outbreak)
BUDGETS = [0, 10, 20, 30, 40, 50, 75, 100, 150, 200]

# Monte-Carlo over seed sets (expected cases averted)
N_SEEDS_PER_RUN = 6      # initial infectious per replicate
N_REPS = 10              # number of replicates for expectation
EPS = 1e-12

# ------------------------------ Curvature & features ---------------------------
class FormanRicciUndirected:
    """Undirected Forman–Ricci curvature with unit weights; returns dict keyed by (min,max) edge."""
    def __init__(self, G: nx.Graph):
        if not isinstance(G, nx.Graph) or G.is_directed():
            raise ValueError("FormanRicciUndirected requires an undirected Graph.")
        self.G = G

    def compute(self) -> Dict[Tuple, float]:
        F = {}
        for u, v in self.G.edges():
            we = wu = wv = 1.0
            su = sum(wu / math.sqrt(we * 1.0) for x in self.G.neighbors(u) if x != v)
            sv = sum(wv / math.sqrt(we * 1.0) for y in self.G.neighbors(v) if y != u)
            key = (u, v) if u < v else (v, u)
            F[key] = we * ((wu / we) + (wv / we) - su - sv)
        return F

def standardize_edge_feature(z: Dict[Tuple, float]) -> Dict[Tuple, float]:
    vals = np.array(list(z.values()), dtype=float)
    mu, sd = vals.mean(), vals.std()
    if sd <= 0 or not np.isfinite(sd):
        return {e: 0.0 for e in z}
    return {e: (v - mu) / sd for e, v in z.items()}

def edge_feature_frc(G: nx.Graph) -> Dict[Tuple, float]:
    return FormanRicciUndirected(G).compute()

def edge_feature_edge_betweenness(G: nx.Graph) -> Dict[Tuple, float]:
    eb = nx.edge_betweenness_centrality(G, normalized=True)
    return {tuple(sorted(k)): v for k, v in eb.items()}

# Node scores used for vaccination targeting
def node_score_from_negative_frc(G: nx.Graph, F: Dict[Tuple, float]) -> Dict[int, float]:
    """s(u) = sum over incident edges of max(0, -F(edge))."""
    s = {u: 0.0 for u in G.nodes()}
    for (u, v), f in F.items():
        w = max(0.0, -float(f))
        s[u] += w
        s[v] += w
    return s

def node_score_from_edge_betweenness(G: nx.Graph, EB: Dict[Tuple, float]) -> Dict[int, float]:
    """Aggregate incident edge betweenness per node."""
    s = {u: 0.0 for u in G.nodes()}
    for (u, v), w in EB.items():
        s[u] += float(w)
        s[v] += float(w)
    return s

# ------------------------------ Build B and simulate ---------------------------
def build_B_from_feature(G: nx.Graph, z_e_raw: Dict[Tuple, float], beta0: float, alpha: float = 1.0):
    z = standardize_edge_feature(z_e_raw)
    nodes = list(G.nodes())
    idx = {v: i for i, v in enumerate(nodes)}
    n = len(nodes)
    B = np.zeros((n, n), dtype=np.float64)
    for u, v in G.edges():
        key = tuple(sorted((u, v)))
        w = beta0 * math.exp(alpha * z.get(key, 0.0))
        i, j = idx[u], idx[v]
        B[i, j] = w
        B[j, i] = w
    np.fill_diagonal(B, 0.0)
    return B, nodes

def simulate_sir_euler_stable(B: np.ndarray,
                              seeds_idx: List[int],
                              immunized_idx: List[int],
                              tvec: np.ndarray,
                              gamma: float = 0.1) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Stable explicit Euler with clamping/renormalization."""
    n = B.shape[0]
    dt = float(tvec[1] - tvec[0])
    T = len(tvec)
    S = np.ones((T, n), dtype=np.float64)
    I = np.zeros((T, n), dtype=np.float64)
    R = np.zeros((T, n), dtype=np.float64)

    # immunize (R=1 at t=0)
    if immunized_idx:
        S[0, immunized_idx] = 0.0
        R[0, immunized_idx] = 1.0

    # infect seeds not immunized
    for i in seeds_idx:
        if i not in immunized_idx:
            S[0, i] = 0.0
            I[0, i] = 1.0

    for t in range(T - 1):
        lam = B @ I[t]
        dS = -S[t] * lam
        dI = S[t] * lam - gamma * I[t]
        dR = gamma * I[t]
        S[t+1] = np.clip(S[t] + dt * dS, 0.0, 1.0)
        I[t+1] = np.clip(I[t] + dt * dI, 0.0, 1.0)
        R[t+1] = np.clip(R[t] + dt * dR, 0.0, 1.0)
        total = S[t+1] + I[t+1] + R[t+1] + EPS
        S[t+1] /= total; I[t+1] /= total; R[t+1] /= total
    return S, I, R

# ------------------------------ Seeds (negative-FRC bridges) -------------------
def pick_seeds_negfrc_maxmin(G: nx.Graph, k: int, frc: Dict[Tuple, float]) -> List[int]:
    """Pick k seed nodes among endpoints of negative-FRC edges, maximally spaced."""
    if k <= 0: return []
    neg_nodes = set()
    for (u, v), f in frc.items():
        if f < 0:
            neg_nodes.add(u); neg_nodes.add(v)
    cand = list(neg_nodes) if neg_nodes else list(G.nodes())
    # start from node with largest incident edge-betweenness sum
    eb = edge_feature_edge_betweenness(G)
    by_bridge = sorted(
        cand,
        key=lambda u: sum(eb.get(tuple(sorted((u, v))), 0.0) for v in G.neighbors(u)),
        reverse=True
    )
    chosen = [by_bridge[0] if by_bridge else cand[0]]
    while len(chosen) < k:
        best_u, best_d = None, -1
        for u in cand:
            if u in chosen: continue
            d_min = np.inf
            for v in chosen:
                try:
                    d = nx.shortest_path_length(G, u, v)
                    d_min = min(d_min, d)
                except nx.NetworkXNoPath:
                    continue
            d_min = 0 if not np.isfinite(d_min) else d_min
            if d_min > best_d:
                best_d, best_u = d_min, u
        if best_u is None:
            best_u = random.choice([x for x in G.nodes() if x not in chosen])
        chosen.append(best_u)
    return chosen

# ------------------------------ Graphs -----------------------------------------
def make_graphs():
    return {
        "ER":  nx.erdos_renyi_graph(N, P_ER, seed=SEED),
        "WS":  nx.watts_strogatz_graph(N, WS_K, WS_P, seed=SEED),
        "BA":  nx.barabasi_albert_graph(N, BA_M, seed=SEED),
        "PLC": nx.powerlaw_cluster_graph(N, PLC_M, PLC_P, seed=SEED),
    }

# ------------------------------ Threshold calibration FIX ----------------------
def calibrate_beta0_supercritical(G: nx.Graph,
                                  z_truth: Dict[Tuple, float],
                                  gamma: float,
                                  base_beta0: float,
                                  s0: float = 1.0,
                                  margin: float = 1.05) -> float:
    """
    Ensure early growth: s0 * rho(B(beta0)) > gamma.
    Compute rho for beta0=1, then choose beta_min = (gamma * margin) / (s0 * rho_unit).
    Return max(base_beta0, beta_min).
    """
    B_unit, _ = build_B_from_feature(G, z_truth, beta0=1.0, alpha=ALPHA)
    rho_unit = float(np.max(np.linalg.eigvals(B_unit).real))
    if rho_unit <= 0:
        return base_beta0  # pathological; leave as is
    beta_min = (gamma * margin) / max(1e-12, s0 * rho_unit)
    return float(max(base_beta0, beta_min))

# ------------------------------ Ablation core ---------------------------------
def expected_cases_averted_vs_budget(G: nx.Graph,
                                     beta0_true: float,
                                     gamma: float,
                                     budgets: List[int],
                                     n_reps: int = 10,
                                     n_seeds: int = 6):
    """
    Returns dict:
      {
        'B': budgets,
        'FRC_mean': [...],
        'FRC_std':  [...],
        'EB_mean':  [...],
        'EB_std':   [...],
        'N': number of nodes
      }
    Cases averted are expressed as FRACTION of population (multiply by N for counts).
    """
    # Hidden truth contact matrix (FRC)
    z_truth = edge_feature_frc(G)

    # FIX: bump beta0 if needed so the outbreak is supercritical; avoids flat zero curves.
    beta0_adj = calibrate_beta0_supercritical(G, z_truth, gamma, base_beta0=beta0_true, margin=1.10)

    B_truth, nodes = build_B_from_feature(G, z_truth, beta0_adj, ALPHA)
    n = len(nodes)

    # Precompute node scores for targeting (fixed across reps)
    F = z_truth
    EB = edge_feature_edge_betweenness(G)
    score_frc = node_score_from_negative_frc(G, F)
    score_eb  = node_score_from_edge_betweenness(G, EB)

    order_frc = [u for u, _ in sorted(score_frc.items(), key=lambda kv: -kv[1])]
    order_eb  = [u for u, _ in sorted(score_eb.items(),  key=lambda kv: -kv[1])]

    # Collect results per budget
    fracs_averted_frc = {B: [] for B in budgets}
    fracs_averted_eb  = {B: [] for B in budgets}

    for r in range(n_reps):
        # Different seed set per replicate (based on FRC bridges)
        seeds_nodes = pick_seeds_negfrc_maxmin(G, n_seeds, F)
        seeds_idx = [nodes.index(u) for u in seeds_nodes]

        # Baseline (no vaccination)
        _, _, R0 = simulate_sir_euler_stable(B_truth, seeds_idx, immunized_idx=[], tvec=TIME, gamma=gamma)
        Rinf0 = float(R0[-1].mean())  # fraction infected

        for B in budgets:
            # FRC-targeted vaccination (exclude seeds from vaccination to mimic pre-epidemic rollout)
            vacc_frc = [u for u in order_frc if u not in seeds_nodes][:B]
            imm_idx = [nodes.index(u) for u in vacc_frc]
            _, _, Rf = simulate_sir_euler_stable(B_truth, seeds_idx, immunized_idx=imm_idx, tvec=TIME, gamma=gamma)
            Rinf_f = float(Rf[-1].mean())
            fracs_averted_frc[B].append(max(0.0, Rinf0 - Rinf_f))

            # Edge-Betweenness-targeted vaccination
            vacc_eb = [u for u in order_eb if u not in seeds_nodes][:B]
            imm_idx = [nodes.index(u) for u in vacc_eb]
            _, _, Re = simulate_sir_euler_stable(B_truth, seeds_idx, immunized_idx=imm_idx, tvec=TIME, gamma=gamma)
            Rinf_e = float(Re[-1].mean())
            fracs_averted_eb[B].append(max(0.0, Rinf0 - Rinf_e))

    B_list = list(budgets)
    frc_mean = [np.mean(fracs_averted_frc[B]) for B in B_list]
    frc_std  = [np.std( fracs_averted_frc[B]) for B in B_list]
    eb_mean  = [np.mean(fracs_averted_eb[B])  for B in B_list]
    eb_std   = [np.std( fracs_averted_eb[B])  for B in B_list]

    return {
        "B": B_list,
        "FRC_mean": frc_mean, "FRC_std": frc_std,
        "EB_mean": eb_mean,   "EB_std": eb_std,
        "N": n,
        "beta0_used": beta0_adj
    }

# ------------------------------ Runner + plots --------------------------------
def run_ablation_all():
    graphs = make_graphs()
    for name, G in graphs.items():
        print(f"\n=== {name}: n={G.number_of_nodes()}, m={G.number_of_edges()} ===")
        res = expected_cases_averted_vs_budget(
            G, beta0_true=BETA0_TRUE, gamma=GAMMA,
            budgets=BUDGETS, n_reps=N_REPS, n_seeds=N_SEEDS_PER_RUN
        )
        B = np.array(res["B"], dtype=float)

        # Plot (fraction and counts axis)
        plt.figure(figsize=(7.5, 4.8), dpi=120)
        # FRC
        plt.plot(B, res["FRC_mean"], label="FRC targeting", color="tab:blue", lw=2)
        plt.fill_between(B,
                         np.array(res["FRC_mean"]) - np.array(res["FRC_std"]),
                         np.array(res["FRC_mean"]) + np.array(res["FRC_std"]),
                         color="tab:blue", alpha=0.15)
        # EdgeBetweenness
        plt.plot(B, res["EB_mean"], label="EdgeBetweenness targeting", color="tab:purple", lw=2, ls="--")
        plt.fill_between(B,
                         np.array(res["EB_mean"]) - np.array(res["EB_std"]),
                         np.array(res["EB_mean"]) + np.array(res["EB_std"]),
                         color="tab:purple", alpha=0.15)
        plt.xlabel("Vaccination budget B (people)")
        plt.ylabel("Expected cases averted (fraction of N)")
        plt.title(f"{name}: cases averted vs budget (truth = FRC, β0={res['beta0_used']:.3f})")
        plt.grid(alpha=0.3, ls=":")
        plt.legend(frameon=False)
        plt.tight_layout()
        out = os.path.join(OUTDIR, f"ablation_cases_averted_{name}.png")
        plt.savefig(out, dpi=300)
        plt.close()
        print(f"Saved: {out}")

        # Also print a small table (counts)
        Npop = res["N"]
        print("Budget B | FRC mean±sd (counts) | EdgeBetw. mean±sd (counts)")
        for Bk, m1, s1, m2, s2 in zip(res["B"], res["FRC_mean"], res["FRC_std"], res["EB_mean"], res["EB_std"]):
            c1m, c1s = m1 * Npop, s1 * Npop
            c2m, c2s = m2 * Npop, s2 * Npop
            print(f"{Bk:7d} | {c1m:8.1f} ± {c1s:6.1f} | {c2m:8.1f} ± {c2s:6.1f}")

# ------------------------------ Main ------------------------------------------
if __name__ == "__main__":
    run_ablation_all()



=== ER: n=1000, m=1469 ===
Saved: images\ablation_cases_averted_ER.png
Budget B | FRC mean±sd (counts) | EdgeBetw. mean±sd (counts)
      0 |      0.0 ±    0.0 |      0.0 ±    0.0
     10 |     11.0 ±    0.0 |      2.8 ±    0.0
     20 |     15.9 ±    0.0 |     11.7 ±    0.0
     30 |     22.6 ±    0.0 |     25.9 ±    0.0
     40 |     30.4 ±    0.0 |     36.2 ±    0.0
     50 |     34.4 ±    0.0 |     41.6 ±    0.0
     75 |     62.2 ±    0.0 |     58.5 ±    0.0
    100 |     89.2 ±    0.0 |     87.2 ±    0.0
    150 |    165.1 ±    0.0 |    145.2 ±    0.0
    200 |    258.0 ±    0.0 |    226.2 ±    0.0

=== WS: n=1000, m=5000 ===
Saved: images\ablation_cases_averted_WS.png
Budget B | FRC mean±sd (counts) | EdgeBetw. mean±sd (counts)
      0 |      0.0 ±    0.0 |      0.0 ±    0.0
     10 |      0.0 ±    0.0 |      0.0 ±    0.0
     20 |      0.0 ±    0.0 |      0.0 ±    0.0
     30 |      0.0 ±    0.0 |      0.0 ±    0.0
     40 |      0.0 ±    0.0 |      0.0 ±    0.0
     50 |     